In [9]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

files = {
    "casseau_qk": "/home/hhoechter/tum/jaxfluids_internship/data/casseau_qk_reactions.json",
    "park": "/home/hhoechter/tum/jaxfluids_internship/data/park_reactions.json",
    "scanlon": "/home/hhoechter/tum/jaxfluids_internship/data/scanlon_table2_reactions.json",
    "park1990": "/home/hhoechter/tum/jaxfluids_internship/data/park_1990_reactions.json",
}


def load_rates(path):
    data = json.loads(Path(path).read_text())
    rates = {}
    for rxn in data.get("reactions", []):
        eq = rxn.get("equation", "")
        rates[eq] = {
            "C_f": rxn.get("C_f"),
            "n_f": rxn.get("n_f"),
            "E_f_over_k": rxn.get("E_f_over_k"),
        }
    return rates


rates_by_source = {name: load_rates(path) for name, path in files.items()}

# Union of reactions (preserve order from first source)
order = []
seen = set()
for src in files.keys():
    for eq in rates_by_source[src]:
        if eq not in seen:
            order.append(eq)
            seen.add(eq)

sources = list(files.keys())


def build_df(param):
    rows = []
    for eq in order:
        row = {
            src: rates_by_source[src].get(eq, {}).get(param, np.nan)
            for src in sources
        }
        rows.append(row)
    return pd.DataFrame(rows, index=order)


df_A    = build_df("C_f")
df_beta = build_df("n_f")

# E_f_over_k: single column — take first non-nan value across sources
ek_vals = []
for eq in order:
    val = np.nan
    for src in sources:
        v = rates_by_source[src].get(eq, {}).get("E_f_over_k")
        if v is not None and np.isfinite(float(v)):
            val = float(v)
            break
    ek_vals.append(val)


def luminance(r, g, b):
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def viridis_cell_colors(df, log_scale=False):
    vals = df.values.astype(float)
    if log_scale:
        vals = np.where(vals > 0, np.log10(vals), np.nan)

    mask = np.isfinite(vals)
    if mask.sum() == 0:
        return [[["#f0f0f0"] * df.shape[1]] * df.shape[0]], [
            [["black"] * df.shape[1]] * df.shape[0]
        ]

    vmin, vmax = np.nanmin(vals[mask]), np.nanmax(vals[mask])
    norm = np.where(
        np.isfinite(vals),
        0.5 if np.isclose(vmin, vmax) else (vals - vmin) / (vmax - vmin),
        np.nan,
    )

    colors, fonts = [], []
    for i in range(df.shape[0]):
        row_c, row_f = [], []
        for j in range(df.shape[1]):
            n = norm[i, j]
            if not np.isfinite(n):
                row_c.append("#f0f0f0")
                row_f.append("black")
            else:
                r = int(68 + 187 * n)
                g = int(1 + 180 * n)
                b = int(84 + 20 * (1 - n))
                row_c.append(f"rgb({r},{g},{b})")
                row_f.append("white" if luminance(r, g, b) < 140 else "black")
        colors.append(row_c)
        fonts.append(row_f)
    return colors, fonts


def viridis_1d(vals_raw, log_scale=False):
    vals = np.array([float(v) for v in vals_raw])
    if log_scale:
        vals = np.where(vals > 0, np.log10(vals), np.nan)
    mask = np.isfinite(vals)
    vmin, vmax = (np.nanmin(vals[mask]), np.nanmax(vals[mask])) if mask.sum() > 1 else (0, 1)
    norm = np.where(np.isfinite(vals), (vals - vmin) / (vmax - vmin) if not np.isclose(vmin, vmax) else 0.5, np.nan)
    colors, fonts = [], []
    for n in norm:
        if not np.isfinite(n):
            colors.append("#f0f0f0"); fonts.append("black")
        else:
            r = int(68 + 187 * n); g = int(1 + 180 * n); b = int(84 + 20 * (1 - n))
            colors.append(f"rgb({r},{g},{b})")
            fonts.append("white" if luminance(r, g, b) < 140 else "black")
    return colors, fonts


def fmt(x):
    try:
        return "" if not np.isfinite(float(x)) else f"{float(x):.4g}"
    except (TypeError, ValueError):
        return ""


# --- Header colors ---
COLOR_RXN  = "#888888"
COLOR_A    = "#2a6496"   # blue
COLOR_BETA = "#c0392b"   # red
COLOR_EK   = "#27ae60"   # green

header_values = (
    ["Reaction"]
    + [f"A · {src}" for src in sources]
    + [f"beta · {src}" for src in sources]
    + ["E_f/k"]
)
header_colors = (
    [COLOR_RXN]
    + [COLOR_A] * len(sources)
    + [COLOR_BETA] * len(sources)
    + [COLOR_EK]
)

# --- Cell values ---
a_colors,    a_fonts    = viridis_cell_colors(df_A,    log_scale=True)
beta_colors, beta_fonts = viridis_cell_colors(df_beta, log_scale=False)
ek_colors,   ek_fonts   = viridis_1d(ek_vals, log_scale=True)

cell_values = (
    [order]
    + [[fmt(df_A[src].iloc[i])    for i in range(len(order))] for src in sources]
    + [[fmt(df_beta[src].iloc[i]) for i in range(len(order))] for src in sources]
    + [[fmt(v) for v in ek_vals]]
)

cell_colors = (
    [["#f7f7f7"] * len(order)]
    + [[a_colors[i][j]    for i in range(len(order))] for j in range(len(sources))]
    + [[beta_colors[i][j] for i in range(len(order))] for j in range(len(sources))]
    + [ek_colors]
)
cell_fonts = (
    [["black"] * len(order)]
    + [[a_fonts[i][j]    for i in range(len(order))] for j in range(len(sources))]
    + [[beta_fonts[i][j] for i in range(len(order))] for j in range(len(sources))]
    + [ek_fonts]
)

n_src = len(sources)
RXN_W = 176
col_widths = [RXN_W] + [80] * n_src + [60] * n_src + [80]

fig = go.Figure(go.Table(
    columnwidth=col_widths,
    header=dict(
        values=header_values,
        fill_color=header_colors,
        align="left",
        font=dict(size=11, color="white"),
        height=30,
    ),
    cells=dict(
        values=cell_values,
        fill_color=cell_colors,
        font=dict(size=10, color=cell_fonts),
        align="left",
        height=18,
    ),
))

fig.update_layout(
    height=max(600, 18 * len(order) + 80),
    width=RXN_W + 80 * n_src + 60 * n_src + 80 + 40,
    margin=dict(l=10, r=10, t=10, b=10),
)
fig.show()
fig.write_image("/home/hhoechter/tum/jaxfluids_internship/experiments/debug/rate_comparison_table.pdf")